# Beat the Bytes · ECG to ESP32
MassAI · November 18, 2026

Run a tiny neural network in Python and compare it with actual C++ inference on a simulated ESP32. This optional notebook covers the deployment demonstration.

Use a CPU runtime. Run setup to download the fixed public workshop files. For an offline run, put `model-golf-data.zip` beside this notebook. If a download fails in Colab, download `model-golf-data.zip` from the README and use the manual upload option in setup. NumPy 2.2.6 was used for the reference build.

In [ ]:
#@title Setup — load the workshop files
from pathlib import Path
from zipfile import ZipFile
from urllib.request import urlopen
from hashlib import sha256
import sys

manual_upload = False  # Set True only if the automatic download fails.
archive_name = 'model-golf-data.zip'
source = 'https://raw.githubusercontent.com/bvetsa/MassAI_Harware_Workshop/7241b01e74d104842c6068d1a73c60e8fcce164d/'
checksums = {'classroom.npz': '658019ee9239fabe8ed9a491ec397f33421e5d1e3537d79889c46f1fae7a6681', 'baseline.mgf': 'f01ba0e654aa7c44e35d94ba8309807d90277f7949c71060a6af7ed200f5788f', 'model_golf.py': '1250c872a6e2f9f3a8708672cba0d62e036be6595415eabe55e8f54eba420107', 'rules.json': 'da534554b4dd8e344e8985c3b5d0b7a88b3ecc2ce0b388b232735b0ca31ec6a6', 'demo_cases.npz': '1f7661d8fc5f43773fd81bd49651768c449898ea2e0a93987248e9bb2ff4d38c', 'wokwi/diagram.json': '2fa2f313032f297d1f6feed0701712095007f37422ea3f6bc56465c6ece754c3', 'wokwi/src/main.cpp': 'ed803928837a29f2e254964c9a28cebee212cb8f0eb35c315e75b3dded0c940f', 'wokwi/src/inference.h': '7ed743835aa08a02cc00faa6fb97964665517b180f73f87b7b4d0635b8fef2b6'}
base = Path('.')
if not all((base / name).is_file() for name in checksums):
    base = Path('model-golf-data')
    if manual_upload and not Path(archive_name).is_file():
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError(f'Put {archive_name} beside this notebook, then rerun setup.') from error
        print(f'Choose {archive_name} from the student kit.')
        uploaded = files.upload()
        if archive_name not in uploaded:
            raise ValueError(f'Please choose {archive_name}.')
        Path(archive_name).write_bytes(uploaded[archive_name])
    for name, digest in checksums.items():
        target = base / name
        if target.is_file() and sha256(target.read_bytes()).hexdigest() == digest:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        if Path(archive_name).is_file():
            with ZipFile(archive_name) as archive:
                content = archive.read(name)
        else:
            try:
                with urlopen(source + name, timeout=30) as response:
                    content = response.read()
            except OSError as error:
                raise RuntimeError('Download failed. Rerun setup. If it keeps failing, download model-golf-data.zip '
                                   'from the README, set manual_upload = True, and rerun setup.') from error
        if sha256(content).hexdigest() != digest:
            raise ValueError(f'{name} differs from the workshop version. Use the current student kit.')
        target.write_bytes(content)

import numpy as np
sys.path.insert(0, str(base.resolve()))
from model_golf import (BASELINE_WIDTH, INPUTS, SEED, TRAINING_EPOCHS, TRAINING_SEEDS,
                        WIDTHS, LABELS, decode, encode, format_score, score, train,
                        browser_sketch, firmware_headers, logits)

with np.load(base / 'classroom.npz', allow_pickle=False) as data:
    x_train, y_train = data['train_x'], data['train_y']
    x_val, y_val = data['validation_x'], data['validation_y']
baseline = (base / 'baseline.mgf').read_bytes()
print(f'Ready · {len(x_train):,} training windows · {len(x_val):,} practice windows · 64 values per beat')

with np.load(base / 'demo_cases.npz', allow_pickle=False) as data:
    cases = {name: data[name] for name in ('id', 'x', 'y')}


## The two labels

Each example is an MLII ECG window around an expert beat annotation in the MIT-BIH Arrhythmia Database. `N` is the database’s normal beat symbol. `V` is a premature ventricular contraction (PVC). The dataset omits other annotation symbols and uses different recordings for training and practice. This classroom model classifies pre-cut windows for those two labels. Clinical use would require a separate validation process.

Start with the included working weights. Accuracy counts every correct prediction. PVC sensitivity counts the share of annotated PVC beats that the model catches.

[MIT-BIH data and license](https://physionet.org/content/mitdb/1.0.0/) · [Beat annotation codes](https://physionet.org/physiotools/wpg/wpg_36.htm)

In [ ]:
baseline_score = score(baseline, x_val, y_val)
print('Pretrained 24-neuron FP32 model')
print(format_score(baseline_score))
print(f'Always-N practice accuracy: {np.mean(y_val == 0):.1%} | PVC sensitivity: 0.0%')

In [ ]:
import matplotlib.pyplot as plt
import hashlib

reference = logits(decode(baseline), cases['x'])
time_ms = (3 * np.arange(64) - 62) * 1000 / 360
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True, layout='constrained')
for ax, beat_id, x, label, z, color in zip(axes, cases['id'], cases['x'], cases['y'],
                                            reference, ('#137e70', '#c4453a')):
    ax.plot(time_ms, x, color=color, linewidth=2)
    ax.axvline(0, color='0.5', linestyle=':')
    ax.set_title(f'{beat_id} · annotated {LABELS[label]}')
    ax.set_xlabel('Milliseconds from annotation')
    print(f'{beat_id} | logit {z:+.8f} | predicted {LABELS[int(z >= 0)]}')
axes[0].set_ylabel('Normalized amplitude')
plt.show()
print('Model SHA-256:', hashlib.sha256(baseline).hexdigest())

## Export the demo

The firmware uses the 64 processed values verbatim, runs both MLP layers, and switches the LEDs from its own logit. A nonnegative logit predicts PVC. The Python reference is embedded for the serial comparison.

Open a [new Wokwi ESP32 project](https://wokwi.com/projects/new/esp32). From the ZIP below, paste `sketch.ino` and `diagram.json` over the corresponding tabs. Start the simulation and look for `parity=PASS` on each beat. The project changes can run without saving to an account.

In [ ]:
import shutil

out = Path('baseline_export')
src = out / 'src'
firmware_headers(baseline, cases, src)
for name in ('main.cpp', 'inference.h'):
    shutil.copyfile(base / 'wokwi' / 'src' / name, src / name)
(out / 'sketch.ino').write_text(browser_sketch(src))
shutil.copyfile(base / 'wokwi' / 'diagram.json', out / 'diagram.json')
(out / 'baseline.mgf').write_bytes(baseline)
with ZipFile('baseline-wokwi.zip', 'w') as export:
    for name in ('baseline.mgf', 'sketch.ino', 'diagram.json'):
        export.write(out / name, name)
print('Saved baseline-wokwi.zip')

In [ ]:
# Run this cell in Colab to download the demo. In Jupyter use the Files panel.
try:
    from google.colab import files
except ImportError:
    print(Path('baseline-wokwi.zip').resolve())
else:
    files.download('baseline-wokwi.zip')

## Optional · Reproduce training

This cell trains from scratch with seed 18. Compare the practice metrics and demo logits. Floating-point libraries on different platforms can produce slightly different trained weights. Use the shipped payload for the reference firmware comparison above.

In [ ]:
retrained = encode(train(x_train, y_train, width=24), bits=32)
print(format_score(score(retrained, x_val, y_val)))
print('Retrained demo logits:', logits(decode(retrained), cases['x']))
Path('retrained-baseline.mgf').write_bytes(retrained)